Cont from `Vizuara` : [Link](https://www.youtube.com/watch?v=xbaYCf2FHSY&list=PLPTV0NXA_ZSgsLAr8YCgCwhPIJNNtexWu&index=5)

Cont from `Sebastian Raschka` : [Tokenization](https://www.youtube.com/watch?v=341Rb8fJxY0&list=PLTKMiZHVd_2IIEsoJrWACkIxLRdfMlw11&index=3)

## **`Pretraining` vs `Fine-tuning`**

### **Pretraining**

The process of training a language model on a large corpus of text data to learn general language patterns and representations. During pretraining, the model is exposed to vast amounts of text and learns to predict the next word in a sentence or fill in missing words based on the surrounding context. This phase helps the model acquire a broad understanding of language structure, grammar, and semantics.

The trained model is also called `Fundamental Model` or `Base Model`.

**No `Labels` are required during Pretraining.**

### **Fine-tuning**

Fine-tuning is the subsequent process of adapting a `pretrained` language model to a specific task or domain. In this phase, the model is further trained on a smaller, task-specific dataset to optimize its performance for a particular application, such as sentiment analysis, question answering, or text classification. Fine-tuning allows the model to leverage the knowledge gained during pretraining while tailoring it to the nuances of the target task.

For example, a model might be trained on all the data available on the internet during pretraining, but during fine-tuning, it could be trained specifically on medical texts to improve its performance in healthcare-related applications.

**`Labels` are required during Fine-tuning.**

So when we fine tune a pretrained model, we need to `Annotated` data for the specific task we are fine tuning for.

<hr>
<hr>


## **Why Don't we Pass `Word Embeddings` Directly to the `Transformer`?**

What is the reason that we don't pass `Word Embeddings` which are of `Fixed Size` directly to the `Transformer` model? Why do we pass the `Embeddings` of the `Tokens` instead?

**Vocabulary Explosion**

So, there are roughly around `170,000` words in the English language. If we're using `1024`-dimensional embeddings, then the size of the embedding matrix would be `170,000 x 1024`. If we use `2 Bytes` per float, then the size of the embedding matrix would be around `340 MB`. This is a huge size for just the embedding layer, and it would require a lot of memory to store and process.

If we use `340MB` just for the embedding layer i.e. `Entry Gate` of the model, then the size of the entire model would be much larger. This would make it difficult to train and deploy the model, especially on devices with limited memory.

**Out of Vocabulary (OOV) Words**

Another reason is that there are many words in the English language that are not included in the vocabulary. These are called `Out of Vocabulary (OOV)` words. If we use fixed-size embeddings, then we would not be able to represent these OOV words. This would lead to a loss of information and would negatively impact the performance of the model.

**Word Embeddings Are Static**

Word embeddings are `static`, meaning that they do not change based on the context in which they are used. But our language is dynamic, and the meaning of words can change based on the context. For example, the word `"bank"` can mean a financial institution or the side of a river, depending on the context. If we use fixed-size embeddings, then we would not be able to capture these nuances in meaning.

**Same Word, Different Forms**

Another challenge with fixed-size embeddings is that they do not account for the different forms of a word. For example, the words `"run"`, `"running"`, and `"ran"` are all different forms of the same word. If we use fixed-size embeddings, then we would need to have separate embeddings for each form of the word. This would further increase the size of the embedding matrix and make it even more difficult to manage.

<hr>

### **Tokenization to the Rescue!**

Now, how do we solve these problems?

One of the most common approaches is to use `Subword Tokenization` techniques like `Byte Pair Encoding (BPE)` or `WordPiece`. 

What if we break down words into smaller units called `Tokens`? But what happens when we break down words into smaller units called `Tokens`?

Suppose we have the word `"unhappiness"`. If we break it down into smaller units, we might get the following tokens: `["un", "happi", "ness"]`. Now, we give a unique `Token ID` to each of these tokens instead of the entire word.

So, instead of having a vocabulary of `170,000` words, we might have a vocabulary of only `30,000` tokens. This significantly reduces the size of the embedding matrix and makes it more manageable.

Once we've this reduced vocabulary of tokens, we then create an `Embedding Matrix` for these tokens.

### **How Do We Generate `Embeddings` for `Tokens`?**

First, we should understand that, **`We don’t use a separate model to generate token embeddings. The embeddings are a part of the LLM itself.`**

When we initialize a `Large Language Model (LLM)`, we also initialize an `Embedding Layer` as part of the model architecture. This `Embedding Layer` is essentially a lookup table that maps each `Token ID` to a corresponding `Embedding Vector`.

Let's understand with an example, say we've `30k` Tokens in our vocabulary and we want to use `1024`-dimensional embeddings. So, what happens is a matrix of size `30,000 x 1024` i.e. a matrix with `30,000` rows and `1024` columns is initialized with random values usually drawn from a normal distribution i.e. $$ \mathcal{N}(0, 0.02) $$.

Each `Row` in this matrix represents the `Embedding Vector` for a specific `Token`. 

If for the word `"unhappiness"`, the tokens are `["un", "happi", "ness"]` and their corresponding `Token IDs` are `[101, 202, 303]`, then during the forward pass of the model, we look up the rows `101`, `202`, and `303` in the embedding matrix to get their respective embedding vectors.

Such as:

$$
\begin{bmatrix}
e_{11} & e_{12} & e_{13} & ... & e_{1,1024} \\
e_{21} & e_{22} & e_{23} & ... & e_{2,1024} \\
e_{31} & e_{32} & e_{33} & ... & e_{3,1024} \\
... & ... & ... & ... & ... \\
e_{30000,1} & e_{30000,2} & e_{30000,3} & ... & e_{30000,1024} \\
\end{bmatrix}
$$

i.e. 

$$
\begin{bmatrix}
0.01 & -0.02 & 0.03 & ... & 0.005 \\
-0.01 & 0.04 & -0.03 & ... & 0.002 \\
0.02 & -0.01 & 0.01 & ... & -0.004 \\
... & ... & ... & ... & ... \\
0.03 & 0.01 & -0.02 & ... & 0.006 \\
\end{bmatrix}
$$

Where each row corresponds to a token's embedding vector.

Then, for the word `"unhappiness"`, we would retrieve the embedding vectors for the tokens `"un"`, `"happi"`, and `"ness"` from the embedding matrix and create a `Matrix` of size `3 x 1024` (since there are `3` tokens) to represent the entire word.

Mathematically,

$$
\text{Embedding Matrix for "unhappiness"} =
\begin{bmatrix}
e_{101,1} & e_{101,2} & e_{101,3} & ... & e_{101,1024} \\
e_{202,1} & e_{202,2} & e_{202,3} & ... & e_{202,1024} \\
e_{303,1} & e_{303,2} & e_{303,3} & ... & e_{303,1024} \\
\end{bmatrix}
$$

This would be the input representation of the word `"unhappiness"` that is fed into the `Transformer` model.

It is the same for the entire text input. The text is tokenized into tokens, and for each token, we look up its embedding vector from the embedding matrix to create the input representation for the model.

**Note:**

- The initial `Embedding Matrix` is randomly initialized, so it does not represent any meaningful information about the tokens at the beginning. When we start training the model, these embeddings are updated through `backpropagation` to capture the semantic relationships between tokens based on the training data.

<hr>

During training, as the model processes text data, the values in the embedding matrix are updated through backpropagation. The model learns to adjust the embedding vectors based on the context in which the tokens appear, allowing it to capture semantic relationships between words.

Since we've `Tokenized` the words into smaller units, the model understand the importance of each subword. For example, if the word `un` is used in different words like `unhappiness`, `unfair`, `unbelievable`, the model learns that the prefix `un` generally indicates a negation or opposite meaning.

This way, tokenization helps in reducing the vocabulary size, handling OOV words, capturing context, and managing different word forms effectively.

<hr>